# CMPE 401 Project 1 — VisDrone YOLOv11 (Colab)

Runs the **Part I baseline** (YOLOv11s, 640px) on a free Colab GPU, and serves as the base for the VRAM-heavy Part III/V runs (larger models, `imgsz=1280`) that don't fit on the RTX 3060 laptop. It clones the repo, installs deps, downloads + converts VisDrone, then trains/evaluates with the same scripts used locally.

**Everything is saved to Google Drive automatically** (`MyDrive/cmpe401_project1/`). Training runs on the fast local disk and `train.py --sync-dir` mirrors the run folder into Drive **after every epoch** and once more when training ends. Evaluation, plots and tables are written to Drive in the same cell, and the cell ends with `drive.flush_and_unmount()` so nothing is left in the upload buffer. There is no separate "save" cell to forget.

**Steps:**
1. **Set the runtime to a GPU:** Runtime → Change runtime type → GPU (T4). Cell 2 should then print `CUDA available: True`.
2. Run cells 1→3 in order. Cell 3 should finish with ~457k boxes kept (same as local).
3. Run cell 4 (~4–5 h on a T4). It asks for Drive access once at the start, then trains → evaluates (val + test-dev) → plots → tables, all saved to Drive. **Keep this tab open and active**: free Colab disconnects after ~90 min idle.
4. **If the session dies mid-run:** reconnect, run cells 1–3 again, set `RESUME = True` in cell 4 and run it. It restores `last.pt` from Drive and continues from the last finished epoch.
5. Cell 5 (optional) remounts Drive and lists what was saved.

> **Note:** after a long run, free Colab may rate-limit GPU access for several hours. If you can't get a GPU, wait for the reset, use Colab Pro, or run locally.

In [ ]:
# 1. Clone your repo (replace with your GitHub URL after you create it)
REPO_URL = "https://github.com/eijikuinbii/CMPE401-YOLOv11"
!git clone $REPO_URL project
%cd project

In [ ]:
# 2. Install dependencies
!pip install -q ultralytics
import torch; print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3. Download + convert VisDrone-DET
!python scripts/download_visdrone.py --root datasets/VisDrone
!python scripts/convert_visdrone.py --root datasets/VisDrone

In [ ]:
# 4. Train → evaluate → plot, ALL SAVED TO GOOGLE DRIVE automatically.
#  - training writes to local disk (fast); --sync-dir mirrors the run into Drive after
#    EVERY epoch and again at the end, so at most one epoch is lost if Colab dies
#  - eval / plots / tables are written into Drive by this same cell
#  - drive.flush_and_unmount() at the end forces any buffered writes up to Drive
from google.colab import drive
drive.mount('/content/drive')

RUN_NAME   = 'baseline_s_640'
DRIVE_ROOT = '/content/drive/MyDrive/cmpe401_project1'
RUN_DIR    = f'{DRIVE_ROOT}/runs/{RUN_NAME}'        # Drive copy of the run (also used by cell 5)
LOCAL_RUN  = f'results/runs/{RUN_NAME}'
RESUME     = True   # set True after a disconnect: continues from last.pt saved in Drive

if not RESUME:
    # Fresh run: clear any LOCAL leftover so Ultralytics doesn't rename it to baseline_s_6402.
    # (Local disk only. An older Drive copy is moved aside to <name>_prev_<timestamp>, never deleted.)
    !rm -rf {LOCAL_RUN}

# --- 1) train (mirrored to Drive every epoch) ---
!python scripts/train.py --model yolo11s.pt --imgsz 640 --batch 16 --epochs 100 \
    --name {RUN_NAME} --sync-dir {DRIVE_ROOT}/runs --extra save_period=25 \
    {'--resume' if RESUME else ''}

# --- 2) evaluate on val + test-dev; outputs (incl. metrics.json) go into the Drive run folder ---
!python scripts/evaluate.py --weights {LOCAL_RUN}/weights/best.pt --split val  --imgsz 640 --project {RUN_DIR}
!python scripts/evaluate.py --weights {LOCAL_RUN}/weights/best.pt --split test --imgsz 640 --project {RUN_DIR}

# --- 3) loss/mAP curves + summary table, then copy them to Drive ---
!python scripts/plot_curves.py {LOCAL_RUN}
!python scripts/summarize_runs.py {LOCAL_RUN} --out results/tables/{RUN_NAME}.md
!mkdir -p {DRIVE_ROOT}/figures {DRIVE_ROOT}/tables
!cp -v results/figures/*{RUN_NAME}*.png {DRIVE_ROOT}/figures/
!cp -v results/tables/{RUN_NAME}.* {DRIVE_ROOT}/tables/

# --- 4) push everything to Drive before the session can go away ---
!ls -la {RUN_DIR} {RUN_DIR}/weights
drive.flush_and_unmount()
print('\nDONE. All outputs saved to Google Drive at', DRIVE_ROOT)

In [ ]:
# 5. (Optional) Remount Drive and list what was saved. Works in a fresh session, no GPU needed.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/cmpe401_project1'
!ls -la {DRIVE_ROOT}/runs/* {DRIVE_ROOT}/figures {DRIVE_ROOT}/tables
!cat {DRIVE_ROOT}/runs/baseline_s_640/eval_*/metrics.json